In [1]:
# 1. Wipe the directory if running interactively just to be safe
!rm -rf /kaggle/working/SAT

# 2. Clone repo
!git clone https://github.com/BorgwardtLab/SAT.git /kaggle/working/SAT

# 3. PyTorch 2.x Compatibility Patch
file_path_models = '/kaggle/working/SAT/sat/models.py'
with open(file_path_models, 'r') as file:
    code_models = file.read()
if "batch_first" not in code_models:
    code_models = code_models.replace("self.encoder = GraphTransformerEncoder(encoder_layer, num_layers)", 
                                      "encoder_layer.self_attn.batch_first = False\n        self.encoder = GraphTransformerEncoder(encoder_layer, num_layers)")
    with open(file_path_models, 'w') as file:
        file.write(code_models)

# 4. NOVEL FEATURE INJECTION: Learnable Gated Residual Connection
patch_code = """
        # --- NOVEL GATED RESIDUAL EXTENSION ---
        # x_orig is the original input feature, x is the GNN output
        if not hasattr(self, 'gate_proj'):
            import torch.nn as nn
            # Dynamically attach a gating linear layer to the model
            self.gate_proj = nn.Linear(x.size(-1) * 2, x.size(-1)).to(x.device)
            
        # Compute the gate: sigmoid(W * [x_orig, x_gnn])
        gate = torch.sigmoid(self.gate_proj(torch.cat([x_orig, x], dim=-1)))
        
        # Selectively combine original features with structural features
        x = gate * x + (1 - gate) * x_orig
        # --------------------------------------
        
        if self.pe_projector is not None:
"""

with open(file_path_models, 'r') as file:
    code_models = file.read()
    
# Inject the gate right after the structure extractor finishes
if "NOVEL GATED RESIDUAL EXTENSION" not in code_models:
    # Find where the base feature is stored
    code_models = code_models.replace("x = self.gnn(x, edge_index, edge_attr, degree)", 
                                      "x_orig = x.clone()\n        x = self.gnn(x, edge_index, edge_attr, degree)")
    # Insert the gating logic
    code_models = code_models.replace("if self.pe_projector is not None:", patch_code)
    
    with open(file_path_models, 'w') as file:
        file.write(code_models)

print("Repository cloned and Gated-SAT Architecture injected successfully! ✅")

Cloning into '/kaggle/working/SAT'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 91 (delta 38), reused 71 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 2.63 MiB | 31.33 MiB/s, done.
Resolving deltas: 100% (38/38), done.
Repository cloned and Gated-SAT Architecture injected successfully! ✅


In [2]:
!pip install torch_geometric ogb performer-pytorch einops -q
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.10.0+cu128.html -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 94.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 104.8 MB/s eta 0:00:00


In [3]:
import sys
if '/kaggle/working/SAT' not in sys.path:
    sys.path.insert(0, '/kaggle/working/SAT')

!cd /kaggle/working/SAT && PYTHONPATH=/kaggle/working/SAT python experiments/train_zinc.py \
    --seed 0 \
    --gnn-type mpnn \
    --use-edge-attr \
    --k-hop 3 \
    --se gnn \
    --abs-pe rw \
    --abs-pe-dim 20 \
    --epochs 2000 \
    --batch-size 128 \
    --lr 0.001 \
    --weight-decay 1e-5 \
    --dropout 0.2 \
    --num-layers 6 \
    --num-heads 8 \
    --dim-hidden 64 \
    --outdir /kaggle/working/results_gated_mpnn

Namespace(seed=0, dataset='ZINC', num_heads=8, num_layers=6, dim_hidden=64, dropout=0.2, epochs=2000, lr=0.001, weight_decay=1e-05, batch_size=128, abs_pe='rw', abs_pe_dim=20, outdir='/kaggle/working/results_gated_mpnn/ZINC/seed0/edge_attr/rw_20/mpnn_3_0.2_0.001_1e-05_6_8_64_BN', warmup=5000, layer_norm=False, use_edge_attr=True, edge_dim=32, gnn_type='mpnn', k_hop=3, global_pool='mean', se='gnn', use_cuda=True, batch_norm=True, save_logs=True)
Extracting ../datasets/ZINC/molecules.zip
Processing...
Processing test dataset: 100%|████████████| 1000/1000 [00:00<00:00, 8605.06it/s]
Done!
/kaggle/working/SAT/experiments/train_zinc.py:207: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_dset, batch_size=args.batch_size,
Data(x=[29], edge_index=[2, 64], edge_attr=[64], y=[1, 1], complete_edge_index=[2, 841], degree=[29])
/kaggle/working/SAT/experiments/train_zinc.py:216: UserWarning: 'data.DataLoader' is deprecated, use 'loader.